In [ ]:
from rdflib import Graph, Namespace, RDF, URIRef

DCAT = Namespace("http://www.w3.org/ns/dcat#")
DCT  = Namespace("http://purl.org/dc/terms/")

def extract_catalog_structure(g):
    """
    Extracts full catalog → dataset → related classes structure from a DCAT-US 1.1 graph.
    Returns a nested dictionary suitable for comparison to DCAT-US 3.0 requirements.
    """

    catalogs = list(g.subjects(RDF.type, DCAT.Catalog))
    if not catalogs:
        raise ValueError("No dcat:Catalog found in the file!")

    result = {}

    for catalog in catalogs:
        catalog_uri = str(catalog)
        result[catalog_uri] = {"datasets": []}

        # 1. All datasets linked to this catalog
        dataset_uris = list(g.objects(catalog, DCAT.dataset))

        for ds in dataset_uris:
            ds_entry = {
                "uri": str(ds),
                "properties": {},
                "related_classes": {}
            }

            # -------------------------------------------
            # 2. Collect ALL triples describing the dataset
            # -------------------------------------------

            for p, o in g.predicate_objects(ds):
                pred = str(p)

                # add property to dataset's property list
                ds_entry["properties"].setdefault(pred, [])
                ds_entry["properties"][pred].append(str(o))

                # -------------------------------------------
                # 3. If object is a resource with a class, capture it
                # -------------------------------------------
                if isinstance(o, URIRef):
                    o_classes = list(g.objects(o, RDF.type))
                    for cls in o_classes:
                        cls_uri = str(cls)
                        ds_entry["related_classes"].setdefault(cls_uri, set())
                        ds_entry["related_classes"][cls_uri].add(str(o))

            # Convert related class sets to lists
            for k in list(ds_entry["related_classes"]):
                ds_entry["related_classes"][k] = list(ds_entry["related_classes"][k])

            # Store dataset entry
            result[catalog_uri]["datasets"].append(ds_entry)

    return result
